
# Core Notebook 3 · Market Data, Curves, and FX

Market data in `finstack.core` covers deterministic term structures (discount, forward, hazard, inflation), interpolation/extrapolation policies, FX matrices, and the `MarketContext` aggregator.


## Imports

In [ ]:

from datetime import date

from finstack.core import currency, dates, money
from finstack.core.market_data import term_structures, fx, context, scalars, surfaces, dividends
from finstack.core.math import interp

usd = currency.Currency("USD")


## Discount curves

In [2]:
from finstack.core.dates.daycount import DayCount, DayCountContext, DayCountContextState

usd_dc = DayCount.from_name("ACT_360")
ois_points = [
    (0.0, 1.0),
    (0.5, 0.9975),
    (1.0, 0.9940),
    (2.0, 0.9850),
]
discount_curve = term_structures.DiscountCurve(
    id="USD-OIS",
    base_date=date(2025, 1, 2),
    knots=ois_points,
    day_count=usd_dc,
    interp="linear",
    extrapolation="flat_zero",
    require_monotonic=True,
)
print("DF(0.75):", round(discount_curve.df(0.75), 6))
print("Zero(1y):", round(discount_curve.zero(1.0), 6))
print("Forward(0.5→1.0):", round(discount_curve.forward(0.5, 1.0), 6))
print("DF on 2025-07-01:", discount_curve.df_on_date(date(2025, 7, 1)))


DF(0.75): 0.99575
Zero(1y): 0.006018
Forward(0.5→1.0): -0.00703
DF on 2025-07-01: 0.9975


## Forward, hazard, and inflation curves

In [3]:
from finstack.core.dates.daycount import DayCount, DayCountContext, DayCountContextState

forward_points = [(0.0, 0.052), (1.0, 0.055), (2.0, 0.0575)]
forward_dc = DayCount.from_name("ACT_365F")
forward_curve = term_structures.ForwardCurve(
    id="USD-SOFR-3M",
    tenor_years=0.25,
    knots=forward_points,
    base_date=date(2025, 1, 2),
    day_count=forward_dc,
)
print("Forward rate at 1y:", forward_curve.rate(1.0))

hazard_points = [(1.0, 0.010), (3.0, 0.0125), (5.0, 0.0150)]
hazard_curve = term_structures.HazardCurve(
    id="ACME-HZD",
    base_date=date(2025, 1, 2),
    knots=hazard_points,
    recovery_rate=0.40,
    day_count=forward_dc,
    issuer="ACME",
    currency=usd,
)
print("Survival at 3y:", round(hazard_curve.survival(3.0), 6))
print("Default prob 1→3y:", round(hazard_curve.default_prob(1.0, 3.0), 6))

infl_points = [(0.0, 265.0), (2.0, 275.0), (5.0, 295.0)]
inflation_curve = term_structures.InflationCurve(
    id="US-CPI",
    base_cpi=260.0,
    knots=infl_points,
    interp="linear",
)
print("Inflation rate 0→5y:", round(inflation_curve.inflation_rate(0.0, 5.0), 6))


Forward rate at 1y: 0.055
Survival at 3y: 0.965605
Default prob 1→3y: 0.024444
Inflation rate 0→5y: 0.026923


## Interpolation & extrapolation utilities

In [4]:

interp_styles = [name for name in dir(interp.InterpStyle) if not name.startswith('_')]
extrapolation_policies = [name for name in dir(interp.ExtrapolationPolicy) if not name.startswith('_')]
print("Interpolation styles:", interp_styles)
print("Extrapolation policies:", extrapolation_policies)


Interpolation styles: ['CUBIC_HERMITE', 'FLAT_FWD', 'LINEAR', 'LOG_LINEAR', 'MONOTONE_CONVEX', 'from_name', 'name']
Extrapolation policies: ['FLAT_FORWARD', 'FLAT_ZERO', 'from_name', 'name']


## FX configuration and matrices

In [5]:
eur = currency.Currency("EUR")
jpy = currency.Currency("JPY")
fx_cfg = fx.FxConfig(pivot_currency=usd, enable_triangulation=True)
print("FX config pivot:", fx_cfg.pivot_currency)

fx_matrix = fx.FxMatrix()
fx_matrix.set_quotes([
    (usd, eur, 0.95),
    (usd, jpy, 150.25),
])
fx_matrix.set_quote(eur, jpy, 158.0)
rate_result = fx_matrix.rate(eur, usd, date(2025, 1, 15), policy=fx.FxConversionPolicy.CASHFLOW_DATE)
print("EUR→USD:", rate_result.rate, "triangulated?", rate_result.triangulated)
print("Cache stats (hits, misses):", fx_matrix.cache_stats())


FX config pivot: USD
EUR→USD: 1.0526315789473684 triangulated? False
Cache stats (hits, misses): (3, 0)


## MarketContext aggregation

In [6]:

market_ctx = context.MarketContext()
market_ctx.insert_discount(discount_curve)
market_ctx.insert_forward(forward_curve)
market_ctx.insert_hazard(hazard_curve)
market_ctx.insert_inflation(inflation_curve)
market_ctx.insert_fx(fx_matrix)

print("Curve IDs:", market_ctx.curve_ids())
print("Count by type:", market_ctx.count_by_type())
print("Has FX?", market_ctx.has_fx, "Total objects:", market_ctx.total_objects)


Curve IDs: ['ACME-HZD', 'US-CPI', 'USD-OIS', 'USD-SOFR-3M']
Count by type: {'Hazard': 1, 'Forward': 1, 'Inflation': 1, 'Discount': 1}
Has FX? <built-in method has_fx of finstack.core.market_data.MarketContext object at 0x1058709b0> Total objects: <built-in method total_objects of finstack.core.market_data.MarketContext object at 0x1058709b0>


## Market scalars and time series

Scalar market data (prices, index levels, generic factors) can be modeled as `MarketScalar` and `ScalarTimeSeries` objects, with configurable interpolation over observation dates.


In [7]:
price_series = scalars.ScalarTimeSeries(
    id="ACME-PRICE",
    observations=[
        (date(2025, 1, 2), 100.0),
        (date(2025, 1, 9), 101.5),
        (date(2025, 1, 16), 102.8),
    ],
    currency=usd,
    interpolation=scalars.SeriesInterpolation.from_name("Linear"),
)

unitless_beta = scalars.MarketScalar.unitless(1.2)
spot_price = scalars.MarketScalar.price(
    money.Money(105.25, usd)
)

print("Series id:", price_series.id)
print("Interpolation:", price_series.interpolation.name)
print("Value on 2025-01-13:", round(price_series.value_on(date(2025, 1, 13)), 4))
print("Values on knots:", price_series.values_on([
    date(2025, 1, 2),
    date(2025, 1, 9),
    date(2025, 1, 16),
]))
print("Beta (unitless):", unitless_beta.value)
print("Spot price scalar is_price?", spot_price.is_price)


Series id: ACME-PRICE
Interpolation: linear
Value on 2025-01-13: 102.2429
Values on knots: [100.0, 101.5, 102.8]
Beta (unitless): 1.2
Spot price scalar is_price? True


## Volatility surfaces

Volatility surfaces capture option vol quotes on an expiry × strike grid and provide interpolation helpers (`value`, `value_checked`, `value_clamped`) for downstream pricing models.


In [8]:
expiries = [0.5, 1.0, 2.0]
strikes = [90.0, 100.0, 110.0]
vol_grid = [
    [0.22, 0.20, 0.21],  # 6M
    [0.24, 0.22, 0.23],  # 1Y
    [0.26, 0.24, 0.25],  # 2Y
]

vol_surface = surfaces.VolSurface(
    id="ACME-EQ-VOL",
    expiries=expiries,
    strikes=strikes,
    grid=vol_grid,
)

print("Vol surface id:", vol_surface.id)
print("Grid shape:", vol_surface.grid_shape)
print("Vol(1Y, 100):", vol_surface.value(1.0, 100.0))
print("Vol clamped beyond grid (3Y, 120):", vol_surface.value_clamped(3.0, 120.0))


Vol surface id: ACME-EQ-VOL
Grid shape: (3, 3)
Vol(1Y, 100): 0.22
Vol clamped beyond grid (3Y, 120): 0.25


## Dividend schedules

Equity dividend assumptions can be expressed as `DividendSchedule` objects built via `DividendScheduleBuilder`, mixing cash, yield, and stock dividends on a single underlying.


In [9]:
builder = dividends.DividendScheduleBuilder("ACME-DIVS")
builder.underlying("ACME")
builder.currency(usd)

builder.cash(date(2025, 3, 15), money.Money(1.00, usd))
builder.yield_div(date(2025, 6, 15), 0.02)
builder.stock(date(2025, 9, 15), 0.05)

div_schedule = builder.build()

print("Dividend schedule id:", div_schedule.id)
print("Underlying:", div_schedule.underlying)
print("Currency:", div_schedule.currency)
print("Cash events:", div_schedule.cash_events)


Dividend schedule id: ACME-DIVS
Underlying: ACME
Currency: USD
Cash events: [(datetime.date(2025, 3, 15), Money(amount=1.000000, currency='USD'))]



## Checklist · best practices

- Store curve IDs, day-counts, interpolation/extrapolation policies alongside knot data for reproducibility
- Prefer `FxMatrix` with triangulation enabled to avoid accidental missing pair lookups
- Accumulate all per-scenario data inside a single `MarketContext` and pass that context downstream
- Keep hazard/inflation/base-correlation metadata (issuer, recovery, detachment) explicit in constructors
- Use `ScalarTimeSeries` / `MarketScalar` for scalar time series instead of ad-hoc lists or dictionaries
- Represent option smiles and equity dividends via `VolSurface` and `DividendSchedule` rather than bespoke data shapes
